In [0]:
import requests, json, time
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ---------- 3) RemoteOK ----------
def fetch_remoteok():
    r = requests.get("https://remoteok.com/api",
                      headers={"User-Agent": "job-copilot-capstone"}, timeout=20)
    r.raise_for_status()
    data = r.json()
    return [d for d in data if isinstance(d, dict) and "id" in d]

remoteok_raw = fetch_remoteok()

# snimi sirove JSON-e u Volume (audit trag + fallback ako transformacija pukne)
dbutils.fs.put("/Volumes/job_copilot/raw/landing/remoteok.json", json.dumps(remoteok_raw), overwrite=True)

In [0]:
schema = StructType([
    StructField("source", StringType()),
    StructField("external_id", StringType()),
    StructField("title", StringType()),
    StructField("company", StringType()),
    StructField("location", StringType()),
    StructField("description", StringType()),
    StructField("url", StringType()),
    StructField("salary_min", DoubleType()),
    StructField("salary_max", DoubleType()),
    StructField("remote", BooleanType()),
    StructField("posted_at", StringType()),
])


def normalize_remoteok(d):
    return {
        "source": "remoteok", "external_id": str(d.get("id")),
        "title": d.get("position"), "company": d.get("company"),
        "location": d.get("location") or "Remote",
        "description": d.get("description"), "url": d.get("url"),
        "salary_min": d.get("salary_min"), "salary_max": d.get("salary_max"),
        "remote": True, "posted_at": d.get("date"),
    }

rows = [normalize_remoteok(d) for d in remoteok_raw]

df = spark.createDataFrame(rows, schema=schema)

df_clean = (df
    .filter(F.col("title").isNotNull() & F.col("description").isNotNull())
    .withColumn("description", F.regexp_replace("description", "<[^>]*>", ""))  # skini HTML tagove
    .withColumn("ingested_at", F.current_timestamp())
    .dropDuplicates(["source", "external_id"]))

(df_clean.write.mode("append")
    .format("delta")
    .saveAsTable("job_copilot.clean.job_postings"))

print(df_clean.count(), "novih oglasa upisano")

In [0]:
import psycopg2
from psycopg2.extras import execute_values
lb_host = "ep-frosty-bar-d8f2o7dt.database.us-east-2.cloud.databricks.com"
lb_port = 5432
lb_db = "databricks_postgres"
lb_user = "gogsbogs"

lb_password = dbutils.secrets.get(
    scope="job-copilot",
    key="lakebase-password"
).strip('\x00')
conn = psycopg2.connect(
    host=lb_host,
    port=lb_port,
    dbname=lb_db,
    user=lb_user,
    password=lb_password,
    sslmode="require"
)
print("Successfully connected to Lakebase!")

In [0]:

rows = df_clean.select("source","external_id","title","company","location",
                        "description","url","salary_min","salary_max","remote","posted_at").collect()

records = [(f"{r.source}_{r.external_id}", r.source, r.title, r.company, r.location,
            r.description, r.url, r.salary_min, r.salary_max, r.remote, r.posted_at) for r in rows]

with conn.cursor() as cur:
    execute_values(cur, """
        INSERT INTO job_postings (job_id, source, title, company, location, description, url,
                                   salary_min, salary_max, remote, posted_at)
        VALUES %s
        ON CONFLICT (job_id) DO UPDATE SET synced_at = now()
    """, records)
conn.commit()

Celija ispod se samo jednom pozove da se kreira tabela. Error baca ponovni poziv. Preskociti pri pokretanju vise puta

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS job_copilot.vector;
CREATE TABLE job_copilot.vector.job_postings_src (
    job_id STRING,
    title STRING,
    company STRING,
    description STRING,
    combined_text STRING   -- title + qualifications + description spojeno, za embedding
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
df_clean.printSchema()

In [0]:
from pyspark.sql import functions as F

df_vec = (df_clean
    .withColumn("job_id", F.concat_ws("_", "source", "external_id"))
    .withColumn("combined_text", F.concat_ws(" | ", F.col("title"), F.col("company"), F.col("description")))
    .select("job_id", "title", "company", "description", "combined_text"))

df_vec.write.mode("append").format("delta").saveAsTable("job_copilot.vector.job_postings_src")

In [0]:
from pyspark.sql import functions as F

df_vec = (df_clean
    .withColumn("job_id", F.concat_ws("_", "source", "external_id"))
    .withColumn("combined_text", F.concat_ws(" | ", F.col("title"), F.col("company"), F.col("description")))
    .select("job_id", "title", "company", "description", "combined_text"))

df_vec.write.mode("append").format("delta").saveAsTable("job_copilot.vector.job_postings_src")
df_vec.head(1)

In [0]:
%sql
ALTER TABLE job_copilot.vector.job_postings_src
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

Kod u celiji ispod se izvrsava/interpretira samo jednom. Svaki naredni put baca izuzetak jer resursi vec postoje.

In [0]:
%pip install databricks-vectorsearch
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
vsc.create_endpoint(name="job-copilot-endpoint", endpoint_type="STANDARD")

vsc = VectorSearchClient()
endpoint = vsc.get_endpoint("job-copilot-endpoint")
print(endpoint)
vsc.list_indexes("job-copilot-endpoint")
index = vsc.create_delta_sync_index(
endpoint_name="job-copilot-endpoint",
source_table_name="job_copilot.vector.job_postings_src",
index_name="job_copilot.vector.job_postings_index",
pipeline_type="TRIGGERED",
primary_key="job_id",
embedding_source_column="combined_text",
embedding_model_endpoint_name="databricks-bge-large-en"  # ugrađeni Databricks FM endpoint
)

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
endpoint = vsc.get_endpoint("job-copilot-endpoint")
print(endpoint)
vsc.list_indexes("job-copilot-endpoint")
index = vsc.get_index(
    endpoint_name="job-copilot-endpoint",
    index_name="job_copilot.vector.job_postings_index"
)

In [0]:
status = index.describe()["status"]

print("State:", status["detailed_state"])
print("Ready:", status["ready"])
print("Message:", status["message"])

In [0]:
results = index.similarity_search(
    query_text="remote backend uloga bez zahteva 5+ godina Kubernetes iskustva",
    columns=["job_id", "title", "company", "description"],
    num_results=10
)

In [0]:
import psycopg2

def get_conn():
    return psycopg2.connect(host=lb_host, port=lb_port, dbname=lb_db,
                             user=lb_user, password=lb_password, sslmode="require")

def search_jobs(query: str, user_id: int, top_k: int = 10) -> list[dict]:
    """Semantička pretraga oglasa + osnovno filtriranje po profilu korisnika."""
    hits = index.similarity_search(query_text=query,
                                    columns=["job_id","title","company","description"],
                                    num_results=top_k)
    return hits["result"]["data_array"]

def explain_match(job_id: str, user_id: int) -> str:
    """Vrati objašnjenje zašto oglas odgovara/ne odgovara profilu (LLM poziv nad description + skills)."""
    # povuci opis oglasa i skills korisnika iz Lakebase, prosledi LLM-u da generiše objašnjenje
    ...

def save_job(user_id: int, job_id: str, stage: str = "saved") -> str:
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO applications (user_id, job_id, stage)
                        VALUES (%s, %s, %s)
                        ON CONFLICT DO NOTHING""", (user_id, job_id, stage))
    conn.commit()
    return f"Oglas {job_id} sačuvan u fazi '{stage}'."

def update_pipeline_stage(user_id: int, job_id: str, new_stage: str) -> str:
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute("""UPDATE applications SET stage=%s, updated_at=now()
                        WHERE user_id=%s AND job_id=%s""", (new_stage, user_id, job_id))
    conn.commit()
    return f"Status ažuriran na '{new_stage}'."

def draft_cover_letter(user_id: int, job_id: str) -> str:
    """Povuci resume + opis posla, pošalji LLM-u da napiše pasus za propratno pismo."""
    ...

def log_interview_note(application_id: int, note: str, follow_up_date: str | None) -> str:
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO interview_notes (application_id, note, follow_up_date)
                        VALUES (%s, %s, %s)""", (application_id, note, follow_up_date))
    conn.commit()
    return "Beleška sačuvana."

def find_stale_applications(user_id: int, days: int = 14) -> list[dict]:
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute("""SELECT application_id, job_id, stage, updated_at FROM applications
                        WHERE user_id=%s AND updated_at < now() - interval '%s days'
                        AND stage NOT IN ('rejected','offer')""", (user_id, days))
        return cur.fetchall()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

tools_schema = [
    {"type": "function", "function": {
        "name": "search_jobs",
        "description": "Pretraži oglase semantički na osnovu opisa željene uloge.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}, "user_id": {"type": "integer"}}}}},
    {"type": "function", "function": {
        "name": "save_job",
        "description": "Sačuvaj oglas u pipeline korisnika sa datim statusom.",
        "parameters": {"type": "object", "properties": {
            "user_id": {"type": "integer"}, "job_id": {"type": "string"},
            "stage": {"type": "string"}}}}},
    # ... ostali alati na isti način
]

def call_agent(user_message: str, user_id: int, history: list):
    response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",  # ili drugi dostupan chat model
        messages=history + [{"role": "user", "content": user_message}]
        #tools=tools_schema,
    )
    # ako model vrati tool_call -> pozovi odgovarajuću Python funkciju, vrati rezultat modelu,
    # pa nastavi razgovor (standardni tool-calling loop)
    ...

TODO Lista
